# MICA Sanity Check — Colab

Purpose: confirm MICA's actual DL inference pipeline runs end-to-end (preprocessing → neural network → clustering/backbone tracing → PULCHRA full-atom build) without crashing. **Not** a benchmark run — no ground-truth comparison, no metrics.

Uses MICA's own official Quick Start sample data (EMD-15635 / PDB 8AT6). Note: this entry is literally in MICA's own training set (confirmed against Supplementary Table S6), so a good-looking result here means nothing about generalization — this is pure plumbing verification, not a real test case. Real scoring happens later, on our own leakage-screened 12-entry benchmark set, not this demo.

**Before running**: Runtime → Change runtime type → Hardware accelerator: **GPU (T4)**. If a "High-RAM" / shape option is available, select it — the local (Mac, 16GB RAM) attempt at this exact same sanity check OOM-killed twice on this entry's 512 grid samples (~13GB+ estimated footprint), which is the whole reason we moved this to Colab.

**Disconnection risk**: the full run (preprocessing + DL inference at batch_size=1 + post-processing) takes ~25-45+ minutes. Free-tier Colab disconnects idle/backgrounded sessions, and `/content` is wiped on a full disconnect — losing everything (cloned repo, installed packages, downloaded model/data) and forcing a full restart from cell 1. **Keep the browser tab open and in the foreground for the whole run** rather than switching away or letting the laptop sleep. If it does disconnect, check with `!ls /content/MICA/trained_models/ 2>/dev/null` first — if that's empty, it's a fresh session and needs `Runtime → Run all` from the top; if it still shows the model file, the same backend session survived and you only need to re-run the inference cell.

In [ ]:
# Confirm GPU + RAM before doing anything expensive
import torch, subprocess
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print()
print(subprocess.run(["cat", "/proc/meminfo"], capture_output=True, text=True).stdout.split("\n")[0])

In [ ]:
# Clone MICA
!git clone https://github.com/jianlin-cheng/MICA.git
%cd MICA

In [ ]:
# Python deps. cython installed FIRST and separately — avoids the exact build
# failure we hit locally (freesasa, pulled in transitively by atom3d, fails to
# build via setuptools' legacy on-the-fly cython auto-fetch; pre-installing
# cython directly sidesteps that broken code path entirely).
#
# superpose3d PINNED to 1.1.1 (matching environment.yml) rather than latest --
# modeler.py:202 calls Superpose3D(...)[0][0], assuming a nested return shape
# that breaks against the current PyPI release (1.5.0) with
# "IndexError: invalid index to scalar variable". Found this the hard way
# after a full successful DL inference run, so don't remove this pin.
!pip install -q cython
!pip install -q biopython mrcfile atom3d "superpose3d==1.1.1" scikit-image rotary-embedding-torch natsort wandb open3d

In [ ]:
# Trained model weights (154.7MB, from the paper's Zenodo record)
!curl -sL -o trained_models.tar.gz "https://zenodo.org/records/15756654/files/trained_models.tar.gz?download=1"
!tar -xzf trained_models.tar.gz
!rm trained_models.tar.gz
!ls trained_models/

In [ ]:
# Official Quick Start sample input (21.1MB compressed on Zenodo -- note this
# is NOT the same file as the 272MB `input.tar` sitting locally; that one had
# already been decompressed, this one starts fresh and unpacks to the same
# content). Includes EMD-15635 map, 8AT6 FASTA, and pre-docked AF3 structures
# (the paper's authors bundled fully-docked demo data, so this tests real
# map+AF3 input fusion, not the degraded map-only fallback).
!curl -sL -o input.tar.gz "https://zenodo.org/records/15756654/files/input.tar.gz?download=1"
!tar -xzf input.tar.gz
!rm input.tar.gz
!ls input/15635/

In [ ]:
# PULCHRA -- unlike the Mac (which needed a custom arm64 compile since the
# bundled binaries are Linux x86-64 / 32-bit macOS, neither usable there),
# Colab IS Linux x86-64, so the repo's own bundled binary should just work.
!chmod +x modules/pulchra304/src/pulchra
!modules/pulchra304/src/pulchra 2>&1 | head -5

In [ ]:
# Force batch_size=1 -- utils/predict.py's own _calculate_optimal_batch_size()
# badly underestimates real GPU memory needs (predicted batch=8 fits a ~10GB
# budget on a 14.6GB T4; actual peak usage for ONE such batch was 13.4GB+,
# causing an immediate CUDA OOM on the very first batch). No CLI flag exists
# for this, so patching the source directly is the only way. Confirmed working
# at batch_size=1 (512/512 samples, ~21 min, no OOM).
!sed -i 's/self\.optimal_batch_size = self\._calculate_optimal_batch_size()/self.optimal_batch_size = 1/' utils/predict.py
!grep -n "optimal_batch_size = " utils/predict.py

In [ ]:
# The actual sanity-check run -- README's Quick Start command, --device cuda
# since we have a real GPU here (this is the whole point of moving off the
# local Mac, which has no CUDA at all).
!python run.py \
  -m input/15635/emd_15635.map \
  -f input/15635/8at6.fasta \
  -a input/15635/AF3_structures \
  --run_pulchra --pulchra_path=modules/pulchra304/src/pulchra \
  --resolution=3.7 --device cuda \
  -o output

In [ ]:
# Confirm something was actually produced
!find output -type f | head -20